# 데이터 정보
- pokemon_rawdata.csv : Kaggle에서 csv 파일로 다운로드한 포켓몬 스텟에 대한 데이터
- final_pokemon_popularity.csv : 구글 트렌드에서 크롤링하여 검색 빈도에 따른 인기도 측정 후 정규화
- pokemon_names_multilingual.csv : 포켓몬 fandom에서 국가별 포켓몬 이름 목록 크롤링
- pokemon_merged_names.csv : rawdata와 multilingual를 영어 이름을 기준으로 merge하여 한국어 이름 컬럼 추가
- final_pokemon_data.csv :  pokemon_merged_names에 final_pokemon_popularity추가
- final_pokemon_popularity_no_normal.csv : 아직 정규화하지 않은 인기도 값

In [1]:
import pandas as pd
from pytrends.request import TrendReq
import cloudscraper
from bs4 import BeautifulSoup
import csv
import time
import random
import os
import requests
from tqdm import tqdm

# 포켓몬 도감 이름 크롤링

In [2]:


# 크롤링할 위키 페이지 URL
url = "https://pokemon.fandom.com/ko/wiki/%EA%B5%AD%EA%B0%80%EB%B3%84_%ED%8F%AC%EC%BC%93%EB%AA%AC_%EC%9D%B4%EB%A6%84_%EB%AA%A9%EB%A1%9D"

def crawl_pokemon_table_bypass():
    try:
        # requests 대신 cloudscraper 사용
        scraper = cloudscraper.create_scraper() 
        
        print("페이지에 접근 중...")
        response = scraper.get(url)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # 앞서 확인하신 prettytable 클래스 사용
        table = soup.find('table', {'class': 'prettytable'})
        
        if not table:
            print("테이블을 찾을 수 없습니다.")
            return

        filename = "pokemon_names_multilingual.csv"
        with open(filename, 'w', encoding='utf-8-sig', newline='') as f:
            writer = csv.writer(f)
            
            rows = table.find_all('tr')
            for row in rows:
                cols = row.find_all(['th', 'td'])
                cols_text = [col.get_text(strip=True) for col in cols]
                
                if any(cols_text):
                    writer.writerow(cols_text)
                    
        print(f"✅ 성공적으로 데이터를 추출하여 '{filename}'에 저장했습니다!")

    except Exception as e:
        print(f"❌ 오류가 발생했습니다: {e}")

if __name__ == "__main__":
    crawl_pokemon_table_bypass()

페이지에 접근 중...
✅ 성공적으로 데이터를 추출하여 'pokemon_names_multilingual.csv'에 저장했습니다!


# 포켓몬 Popularity 수집
- 구글 트렌드를 전세계로 지정하고 포켓몬의 한국 이름과 영어 이름을 둘 다 수집 후 더 낮은 popularity를 선정

In [ ]:


def collect_global_advanced_popularity(input_csv, output_csv='pokemon_popularity_final.csv'):
    # 1. 도감 파일 로드
    try:
        df_pokedex = pd.read_csv(input_csv)
        pokemon_pairs = df_pokedex[['한국어', '영어']].dropna().values.tolist()
    except Exception as e:
        print(f"❌ 파일 로드 에러: {e}")
        return

    # 2. 초기화 및 앵커 설정
    pytrends = TrendReq(hl='en-US', tz=360) 
    anchor_kor = '암멍이'
    anchor_eng = 'Rockruff'
    
    results = []
    print(f"🌍 글로벌 인기도 수집 시작 (앵커: {anchor_kor} -> 0.5 고정)")
    print("💡 x/(x+1) 스케일링을 사용하여 모든 수치를 1 미만의 소수로 보존합니다.")

    # 3. 데이터 수집 루프
    for i, (kor_name, eng_name) in enumerate(pokemon_pairs):
        try:
            # --- (A) 한국어 이름 비교 ---
            pytrends.build_payload([anchor_kor, kor_name], timeframe='today 12-m', geo='')
            data_kor = pytrends.interest_over_time()
            score_kor = 0.0
            if not data_kor.empty and anchor_kor in data_kor.columns:
                m_anchor = data_kor[anchor_kor].mean()
                m_target = data_kor[kor_name].mean()
                score_kor = m_target / m_anchor if m_anchor > 0 else 0

            time.sleep(6)

            # --- (B) 영어 이름 비교 ---
            pytrends.build_payload([anchor_eng, eng_name], timeframe='today 12-m', geo='')
            data_eng = pytrends.interest_over_time()
            score_eng = 0.0
            if not data_eng.empty and anchor_eng in data_eng.columns:
                m_anchor = data_eng[anchor_eng].mean()
                m_target = data_eng[eng_name].mean()
                score_eng = m_target / m_anchor if m_anchor > 0 else 0

            # --- (C) 최소값 선택 및 Soft-max 스타일 스케일링 ---
            # 노이즈 제거를 위한 최소 배수 선택
            min_r = min(score_kor, score_eng)
            
            # [핵심 로직] x / (x + 1) 함수 적용
            # min_r이 1(암멍이와 동급)이면 1/2 = 0.5
            # min_r이 10(암멍이 10배)이면 10/11 = 0.909...
            # min_r이 무한대여도 1에 수렴할 뿐 1이 되지 않음
            final_popularity = min_r / (min_r + 1) if min_r > 0 else 0
            
            results.append({'Name': kor_name, 'Popularity': round(final_popularity, 6)})
            print(f"[{i+1}/{len(pokemon_pairs)}] ✅ {kor_name}: {final_popularity:.6f}")

            time.sleep(10)

        except Exception as e:
            print(f"❌ {kor_name} 에러: {e}")
            results.append({'Name': kor_name, 'Popularity': 0.0})
            time.sleep(12) # 에러 시 대기 시간 증가

        # 20개마다 백업 (더 자주 백업)
        if (i + 1) % 20 == 0:
            pd.DataFrame(results).to_csv('data/temp_global_backup.csv', index=False, encoding='utf-8-sig')

    # 4. 최종 결과 저장
    df_final = pd.DataFrame(results)
    df_final.to_csv(output_csv, index=False, encoding='utf-8-sig')
    print(f"\n✨ 수집 완료! 최종 파일: {output_csv}")

if __name__ == "__main__":
    # 파일 경로가 다를 경우 r'C:\폴더\파일.csv' 형태로 수정하세요.
    collect_global_advanced_popularity('data/pokemon_names_multilingual.csv')

🌍 글로벌 인기도 수집 시작 (앵커: 암멍이 -> 0.5 고정)
💡 x/(x+1) 스케일링을 사용하여 모든 수치를 1 미만의 소수로 보존합니다.
[1/1025] ✅ 이상해씨: 0.748621
[2/1025] ✅ 이상해풀: 0.135275
[3/1025] ✅ 이상해꽃: 0.656836
[4/1025] ✅ 파이리: 0.814239
[5/1025] ✅ 리자드: 0.648924
[6/1025] ✅ 리자몽: 0.920494
[7/1025] ✅ 꼬부기: 0.806755
[8/1025] ✅ 어니부기: 0.084932
[9/1025] ✅ 거북왕: 0.640977
[10/1025] ✅ 캐터피: 0.171110
[11/1025] ✅ 단데기: 0.084642
[12/1025] ✅ 버터플: 0.424957
[13/1025] ✅ 뿔충이: 0.028986
[14/1025] ✅ 딱충이: 0.005944
[15/1025] ✅ 독침붕: 0.414367
[16/1025] ✅ 구구: 0.549247
[17/1025] ✅ 피죤: 0.431583
[18/1025] ✅ 피죤투: 0.494701
[19/1025] ✅ 꼬렛: 0.154965
[20/1025] ✅ 레트라: 0.226612
[21/1025] ✅ 깨비참: 0.009630
[22/1025] ✅ 깨비드릴조: 0.011038
[23/1025] ✅ 아보: 0.469037
[24/1025] ✅ 아보크: 0.244357
[25/1025] ✅ 피카츄: 0.972233
[26/1025] ✅ 라이츄: 0.784705
[27/1025] ✅ 모래두지: 0.294302
[28/1025] ✅ 고지: 0.357991
[29/1025] ✅ 니드런♀: 0.000000
[30/1025] ✅ 니드리나: 0.006672
[31/1025] ✅ 니드퀸: 0.188599
[32/1025] ✅ 니드런♂: 0.004729
[33/1025] ✅ 니드리노: 0.009623
[34/1025] ✅ 니드킹: 0.448432
[35/1025] ✅ 삐삐: 0.675551
[36/1025] ✅ 픽

# 누락된 값들만 수동으로 값 채워주기

In [3]:


def get_single_pokemon_popularity(target_pokemon, anchor_pokemon='파이리'):
    # pytrends 초기화
    pytrends = TrendReq(hl='ko-KR', tz=540)
    
    print(f"🔍 '{target_pokemon}'의 인기도를 '{anchor_pokemon}'(0.5) 기준으로 측정합니다...")
    
    try:
        # 데이터 요청
        pytrends.build_payload([anchor_pokemon, target_pokemon], timeframe='today 12-m', geo='KR')
        data = pytrends.interest_over_time()
        
        if not data.empty and anchor_pokemon in data.columns and target_pokemon in data.columns:
            anchor_mean = data[anchor_pokemon].mean()
            target_mean = data[target_pokemon].mean()
            
            if anchor_mean > 0:
                # 파이리 대비 배수 계산
                relative_score = target_mean / anchor_mean
                
                # 0.5 곱하고 1.0 이상은 1.0으로 자르기 (기존 메인 코드와 동일 로직)
                final_popularity = min(relative_score * 0.5, 1.0)
                
                print("\n✨ [결과] ✨")
                print(f"이름: {target_pokemon}")
                print(f"상대적 검색량: 파이리의 {relative_score:.2f}배")
                print(f"👉 최종 인기도 (Popularity): {final_popularity:.4f}")
                print("-" * 30)
                
                return final_popularity
            else:
                print(f"⚠️ 기준점인 '{anchor_pokemon}'의 검색 데이터가 비정상입니다.")
        else:
            print(f"⚠️ '{target_pokemon}'에 대한 구글 검색 데이터가 없어 0.0000 으로 처리해야 합니다.")
            return 0.0
            
    except Exception as e:
        print(f"\n❌ 에러 발생: {e}")
        if "429" in str(e):
            print("\n💡 [429 Too Many Requests 해결 팁]")
            print("현재 구글이 IP를 차단한 상태입니다. 계속 돌려도 똑같은 에러가 납니다.")
            print("1. 노트북을 스마트폰 테더링(핫스팟)으로 연결하여 IP를 바꿔보세요.")
            print("2. 집 공유기 전원을 껐다 켜서 IP를 새로 할당받아 보세요.")
            print("3. 위 방법이 안 된다면 1~2시간 정도 기다린 후 다시 시도해야 합니다.")

if __name__ == "__main__":
    # 📝 여기에 에러가 났던 포켓몬 이름을 하나씩 넣고 실행하세요.
    target_name = "캥카" 
    
    get_single_pokemon_popularity(target_name)

🔍 '캥카'의 인기도를 '파이리'(0.5) 기준으로 측정합니다...

✨ [결과] ✨
이름: 캥카
상대적 검색량: 파이리의 0.32배
👉 최종 인기도 (Popularity): 0.1603
------------------------------


# 누락된 값 자동으로 채우기

In [ ]:
def update_missing_popularity(pokedex_csv, popularity_csv):
    # 1. 데이터 로드
    try:
        df_pokedex = pd.read_csv(pokedex_csv)
        # 기존 수집된 결과가 있으면 불러오고, 없으면 새로 만듭니다.
        if os.path.exists(popularity_csv):
            df_existing = pd.read_csv(popularity_csv)
            print(f"📂 기존 파일을 불러왔습니다: {len(df_existing)}마리 데이터 존재")
        else:
            print("❌ 기존 인기도 파일이 없습니다. 먼저 전체 수집을 진행해야 합니다.")
            return

        # 2. 누락된 대상 식별 (Popularity가 0.0인 항목들)
        # 수집 중 에러가 나면 0.0으로 저장되게 설정했으므로 이를 기준으로 필터링합니다.
        missing_list = df_existing[df_existing['Popularity'] == 0.0]['Name'].tolist()
        
        if not missing_list:
            print("✨ 갱신할 데이터가 없습니다. 모든 값이 정상적으로 채워져 있습니다.")
            return

        print(f"🔍 총 {len(missing_list)}마리의 누락 데이터를 발견했습니다. 복구를 시작합니다.")

        # 3. 수집 초기화
        pytrends = TrendReq(hl='en-US', tz=360)
        anchor_kor = '암멍이'
        anchor_eng = 'Rockruff'

        # 4. 루프 돌며 데이터 갱신
        for i, target_kor in enumerate(missing_list):
            try:
                # 영어 이름 찾기
                target_eng = df_pokedex[df_pokedex['한국어'] == target_kor]['영어'].values[0]
                
                # --- (A) 한국어 이름 비교 ---
                pytrends.build_payload([anchor_kor, target_kor], timeframe='today 12-m', geo='')
                data_kor = pytrends.interest_over_time()
                score_kor = 0.0
                if not data_kor.empty and anchor_kor in data_kor.columns:
                    m_anchor = data_kor[anchor_kor].mean()
                    m_target = data_kor[target_kor].mean()
                    score_kor = m_target / m_anchor if m_anchor > 0 else 0

                time.sleep(random.uniform(5, 8)) # 안전한 대기

                # --- (B) 영어 이름 비교 ---
                pytrends.build_payload([anchor_eng, target_eng], timeframe='today 12-m', geo='')
                data_eng = pytrends.interest_over_time()
                score_eng = 0.0
                if not data_eng.empty and anchor_eng in data_eng.columns:
                    m_anchor = data_eng[anchor_eng].mean()
                    m_target = data_eng[target_eng].mean()
                    score_eng = m_target / m_anchor if m_anchor > 0 else 0

                # --- (C) 데이터 계산 ---
                min_r = min(score_kor, score_eng)
                final_val = min_r / (min_r + 1) if min_r > 0 else 0.000001 # 에러와 구분하기 위해 최소값 부여
                
                # 데이터프레임에 값 업데이트
                df_existing.loc[df_existing['Name'] == target_kor, 'Popularity'] = round(final_val, 6)
                
                print(f"[{i+1}/{len(missing_list)}] ✅ {target_kor} 복구 완료: {final_val:.6f}")

                # 🛡️ 매번 저장 (안전 장치)
                df_existing.to_csv(popularity_csv, index=False, encoding='utf-8-sig')
                
                # 다음 요청 전 안전한 대기 (10~15초)
                time.sleep(random.uniform(10, 15))

            except Exception as e:
                print(f"❌ {target_kor} 복구 실패 (여전히 차단 상태일 수 있음): {e}")
                time.sleep(30) # 에러 시 더 길게 휴식

        print(f"\n✨ 모든 복구 작업이 완료되었습니다. 결과가 {popularity_csv}에 업데이트되었습니다.")

    except Exception as e:
        print(f"❌ 작업 중 오류 발생: {e}")

if __name__ == "__main__":
    # 파일명을 본인의 환경에 맞게 수정하세요.
    collect_pokedex = 'data/pokemon_names_multilingual.csv'
    target_popularity = 'pokemon_popularity_final.csv'
    
    update_missing_popularity(collect_pokedex, target_popularity)

📂 기존 파일을 불러왔습니다: 1025마리 데이터 존재
🔍 총 1025마리의 누락 데이터를 발견했습니다. 복구를 시작합니다.
❌ 이상해씨 복구 실패 (여전히 차단 상태일 수 있음): The request failed: Google returned a response with code 429
[2/1025] ✅ 이상해풀 복구 완료: 0.121046
[3/1025] ✅ 이상해꽃 복구 완료: 0.669133
❌ 파이리 복구 실패 (여전히 차단 상태일 수 있음): The request failed: Google returned a response with code 429
[5/1025] ✅ 리자드 복구 완료: 0.647662
[6/1025] ✅ 리자몽 복구 완료: 0.923469
[7/1025] ✅ 꼬부기 복구 완료: 0.817520
[8/1025] ✅ 어니부기 복구 완료: 0.080859
❌ 거북왕 복구 실패 (여전히 차단 상태일 수 있음): The request failed: Google returned a response with code 429
❌ 캐터피 복구 실패 (여전히 차단 상태일 수 있음): The request failed: Google returned a response with code 429
❌ 단데기 복구 실패 (여전히 차단 상태일 수 있음): The request failed: Google returned a response with code 429
[12/1025] ✅ 버터플 복구 완료: 0.455744
[13/1025] ✅ 뿔충이 복구 완료: 0.090050
[14/1025] ✅ 딱충이 복구 완료: 0.024816
[15/1025] ✅ 독침붕 복구 완료: 0.438725


KeyboardInterrupt: 

In [8]:
import pandas as pd
from sklearn.preprocessing import QuantileTransformer

# 1. 데이터 로드
df = pd.read_csv('data/final_pokemon_popularity.csv')

# 2. Quantile Transformer 적용 (정규분포화)
# Popularity 열의 값을 정규분포 형태로 변환하여 덮어씁니다.
qt = QuantileTransformer(output_distribution='normal', n_quantiles=len(df), random_state=42)
df['Popularity'] = qt.fit_transform(df[['Popularity']])

# 3. 'Name'과 'Popularity' 열만 선택 (필요한 경우)
df_result = df[['Name', 'Popularity']]

# 4. CSV 파일로 저장
df_result.to_csv('normalized_pokemon_names_popularity.csv', index=False, encoding='utf-8-sig')

print("이름과 정규화된 인기도가 포함된 'normalized_pokemon_names_popularity.csv' 파일이 저장되었습니다.")

이름과 정규화된 인기도가 포함된 'normalized_pokemon_names_popularity.csv' 파일이 저장되었습니다.


# 한국어 포켓몬 이름 추가

In [ ]:


# 1. 데이터 불러오기
# pokemon_stats.csv: 상세 정보가 담긴 파일
# pokemon_names.csv: 크롤링한 한글/영어 이름 매핑 파일
df_stats = pd.read_csv('./data/pokemon_rawdata.csv', dtype={'dexnum': str}) 
df_names = pd.read_csv('./data/pokemon_names_multilingual.csv')

# 2. 크롤링 데이터 컬럼명 확인 및 정제
# 크롤링한 표의 컬럼명이 '영어명', '한국어명'이라고 가정합니다.
# 만약 다를 경우 본인의 csv 컬럼명에 맞게 수정하세요.
df_names = df_names[['영어', '한국어']] 

# 3. 데이터 병합 (Merge)
# df_stats의 'name' 컬럼과 df_names의 '영어명' 컬럼을 기준으로 합칩니다.
# 'left' 조인을 써야 상세 정보(stats) 데이터가 손실되지 않습니다.
merged_df = pd.merge(df_stats, df_names, left_on='name', right_on='영어', how='left')

# 4. 불필요해진 중복 컬럼 제거 및 순서 정리
# '영어명' 컬럼은 'name'과 중복되므로 삭제합니다.
merged_df = merged_df.drop(columns=['영어'])

# 한글 이름을 맨 앞쪽(예: name 옆)으로 옮기고 싶다면 컬럼 순서를 재배치합니다.
cols = list(merged_df.columns)
# '한국어명'을 찾아서 'name' 바로 뒤로 삽입
ko_idx = cols.index('한국어')
cols.insert(2, cols.pop(ko_idx))
merged_df = merged_df[cols]
merged_df = merged_df.rename(columns={'한국어': 'korean name'})

# 5. 결과 확인 및 저장
print(merged_df.head())
merged_df.to_csv('./data/pokemon_merged_names.csv', index=False, encoding='utf-8-sig')

# 인기도 정보 추가

In [9]:


# 1. 데이터 로드
popularity_df = pd.read_csv('data/final_pokemon_popularity.csv')
merged_names_df = pd.read_csv('data/pokemon_merged_names.csv')

# 2. 'korean name'과 'Name' 컬럼을 기준으로 병합 (Left Join)
# pokemon_merged_names.csv의 모든 데이터를 유지하면서 인기도 정보를 추가합니다.
final_df = pd.merge(
    merged_names_df, 
    popularity_df, 
    left_on='korean name', 
    right_on='Name', 
    how='left'
)

# 3. 병합 후 중복된 'Name' 컬럼은 삭제
if 'Name' in final_df.columns:
    final_df = final_df.drop(columns=['Name'])

# 4. 최종 결과를 final_pokemon_data.csv로 저장
final_df.to_csv('data/final_pokemon_data.csv', index=False)

print("파일 생성이 완료되었습니다: final_pokemon_data.csv")

파일 생성이 완료되었습니다: final_pokemon_data.csv


# 포켓몬의 진화 단계 추가

In [2]:
# 1. 기존 데이터 로드 (경로는 snippet에 명시된 data/ 폴더 기준)
file_path = 'data/final_pokemon_data.csv'
df = pd.read_csv(file_path)

# 2. 수집 효율을 위한 세션 설정 및 함수 정의
session = requests.Session()

def fetch_evolution_info(pokemon_name, sess):
    """
    PokeAPI를 통해 진화 단계(0,1,2)를 반환하는 함수
    """
    try:
        # 이름 전처리 (소문자화 및 특수문자 제거)
        api_name = str(pokemon_name).lower().replace(' ', '-').replace("'", "").replace(".", "")
        
        # Species 정보 확인
        species_url = f"https://pokeapi.co/api/v2/pokemon-species/{api_name}"
        res = sess.get(species_url, timeout=5)
        if res.status_code != 200:
            return 0
            
        species_data = res.json()
        evo_chain_url = species_data['evolution_chain']['url']
        
        # 진화 계통도 정보 확인
        evo_res = sess.get(evo_chain_url, timeout=5)
        evo_data = evo_res.json()
        chain = evo_data['chain']
        
        # 단계 계산 (0: 기본, 1: 1차 진화, 2: 2차 진화)
        if chain['species']['name'] == api_name:
            return 0
        for evo1 in chain['evolves_to']:
            if evo1['species']['name'] == api_name:
                return 1
            for evo2 in evo1['evolves_to']:
                if evo2['species']['name'] == api_name:
                    return 2
        return 0
    except:
        return 0

# 3. 데이터 수집 적용
tqdm.pandas(desc="진화 정보 수집 및 여부 판별 중")
df['Evolution_Stage'] = df['name'].progress_apply(lambda x: fetch_evolution_info(x, session))

# 4. '진화 여부' 파생 변수 추가 (여부 확인용)
# 진화 단계가 1 이상이면 진화한 것으로 간주(1), 0단계면 미진화(0)
df['is_evolved'] = (df['Evolution_Stage'] > 0).astype(int)

# 5. 파일 저장
df.to_csv(file_path, index=False)

print(f"작업 완료: 'Evolution_Stage' 및 'is_evolved' 컬럼이 추가되어 {file_path}에 저장되었습니다.")

진화 정보 수집 및 여부 판별 중: 100%|██████████| 1025/1025 [05:31<00:00,  3.09it/s]

작업 완료: 'Evolution_Stage' 및 'is_evolved' 컬럼이 추가되어 data/final_pokemon_data.csv에 저장되었습니다.
